# NoteMap: Visualizing Domain Knowledge Graphically

# Ingestion

In [1]:
from pathlib import Path
from IPython.display import Image, display

import os
from anthropic import Anthropic
from openai import OpenAI
from dotenv import load_dotenv


from notemap.ingest.pdf import rasterize_pdf, transcribe_page, save_pdfs, ingest_pdf
from notemap.cache import content_hash, cache_path
from notemap.embeddings.chunking import chunks_from_pdf
from notemap.embeddings.embed import chunks_to_embeddings, load_pdf_embeddings
from notemap.clustering.umap import umap_reduce, plot_embeddings, hdb_cluster, plot_embedding_clusters
from notemap.clustering.nodes import create_layout, create_centroid_nodes

%load_ext autoreload
%autoreload 2

/opt/anaconda3/envs/NoteMap/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PDF_IN_PATH = Path("notes/pdfs")
DATA_PATH = Path("data")
manifest_path = DATA_PATH / "embeddings" / "manifest.json"

BATCH_SIZE = 256
SEED = 777
load_dotenv()

ocr_client = Anthropic()
embedding_client = OpenAI()
ocr_model = "claude-haiku-4-5"
embedding_model = "text-embedding-3-small"



In [153]:
save_pdfs(PDF_IN_PATH, DATA_PATH, client=ocr_client, model=ocr_model)

Ingested and saved JSON data for  data/documents/Lecture 2- Optim_945206f5.json
Ingested and saved JSON data for  data/documents/Geometric Optics_e8d0eb63.json
Ingested and saved JSON data for  data/documents/Principle Ray Diagrams_ef6ce560.json
Ingested and saved JSON data for  data/documents/duality continued_7bad7fa3.json
Ingested and saved JSON data for  data/documents/EE120-FA24-DISC02_2d9aa91e.json
Ingested and saved JSON data for  data/documents/pitchbook_prep_77a66636.json
Ingested and saved JSON data for  data/documents/Lecture 7- GDA_0d742f77.json
Ingested and saved JSON data for  data/documents/CNNS_cde2d033.json
Ingested and saved JSON data for  data/documents/Lec8- IAGS, CT State Space_b262eb9b.json
Ingested and saved JSON data for  data/documents/Lecture 13- Convexity_f3001efc.json
Ingested and saved JSON data for  data/documents/HW-10_f9ec0abb.json
Ingested and saved JSON data for  data/documents/AVL Trees_4bff18ad.json
Ingested and saved JSON data for  data/documents/PE

In [9]:
embeddings, chunk_ids = load_pdf_embeddings(DATA_PATH, client=embedding_client, model=embedding_model, batch_size=BATCH_SIZE)
print(embeddings.shape)

(678, 1536)


## UMAP (DIM REDUCTION) and Clustering

#### Reduce from ~1536 dimensions to ~40 for HDBScan -> 2 dimensions for visualizing clusters

In [5]:
UMAP1_PARAMS = {
    "n_neighbors": 15,
    "n_components": 10,
    "min_distance": 0.0
}

HDB_PARAMS = {
    "min_samples": 1,
    "min_cluster_size": 20,
    "method": "eom"
}

UMAP2_PARAMS = {
    "n_neighbors": 20,
    "n_components": 2,
    "min_distance": 0.2
}

In [7]:
create_layout(embeddings, HDB_PARAMS, UMAP1_PARAMS, UMAP2_PARAMS, SEED=SEED)

Unlabeled points: 93/678


In [8]:
reduced_embeddings = umap_reduce(embeddings, n_neighbors=15, n_components=10, min_distance=0.0, seed=SEED)



# ALL PLOTTING DOWN
labels = hdb_cluster(reduced_embeddings, min_samples=1, min_cluster_size=10, method="eom")
reduced_embeddings_2d = umap_reduce(embeddings, n_neighbors=20, n_components=2, min_distance=0.1, seed=SEED)
plot_embedding_clusters(reduced_embeddings_2d, labels, manifest_path)


Unlabeled points: 74/678


In [18]:
plot_embeddings(reduced_embeddings_2d, manifest_path)